Constructing a subject-independent version of the fixed ZSL evaluation protocol for UiS4ADL. The predefined class split from `Experiments.ipynb` is kept unchanged, with 18 seen ADLs and 6 unseen ADLs.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import json
sys.path.append('..')

import numpy as np
import pandas as pd

from Feature_extraction.feature_extractor import FeatureExtractor
from Embeddings.Models.embedding_generator import EmbeddingGenerator
from Embeddings.Semantics.descriptions import get_descriptions
from Model.model import IP_SAE_MODEL
from sklearn.metrics import balanced_accuracy_score, recall_score

In [2]:
# Parameters
FS = 100

num_unseen_subjects = 13 # 30.23% of the total 43 subjects

WINDOW_SECONDS = 4.0
OVERLAP_RATIO = 0.0
METHOD = 'temporal_frequency'
STRATEGY = 'retain_short'

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
DESCRIPTION_TYPE = 'original'
USE_PROMPT = True

LAMBDA = 0.001

N_INFERENCE_RUNS = 35
N_SUBJECT_SPLITS = 50

RANDOM_SEED = 22

# Paths
DATASET_PATH = f"../Data/UiS4ADL/Processed/UiS4ADL_{FS}hz.csv"
ADL_DICT_PATH = '../Data/adl_dict.json'
RESULTS_DIR = "../Results"

# Labels
with open(ADL_DICT_PATH, 'r') as f:
    adl_dict_raw = json.load(f)


In [3]:
print("Loading dataset:")
data = pd.read_csv(DATASET_PATH)
print(f"Loaded dataset: {data.shape}")
print(f"Columns: {list(data.columns)}")
all_classes = sorted(np.unique(data['adl']))

Loading dataset:
Loaded dataset: (8424664, 28)
Columns: ['timestamp', 'accX[mg]', 'accY[mg]', 'accZ[mg]', 'gyroX[mdps]', 'gyroY[mdps]', 'gyroZ[mdps]', 'magnX[mG]', 'magnY[mG]', 'magnZ[mG]', 'fxQ1', 'fxQ2', 'fxQ3', 'fxQ4', 'fxYaw[deg]', 'fxPitch[deg]', 'fxRoll[deg]', 'fxGravityX[g]', 'fxGravityY[g]', 'fxGravityZ[g]', 'fxLinAccX[g]', 'fxLinAccY[g]', 'fxLinAccZ[g]', 'fxHeading[deg]', 'adl', 'session', 'subject', 'fileID']


In [4]:
# From the split ranking in Experiments.ipynb:
seen_adls = [7, 12, 10, 8, 6, 21, 23, 20, 22, 15, 13, 1, 19, 4, 16, 2, 18, 3]
unseen_adls = [17, 14, 11, 9, 24, 5]

assert set(seen_adls).isdisjoint(unseen_adls)
assert set(seen_adls + unseen_adls) == set(all_classes)

## Semantic embeddings

In [ ]:
print("Loading semantic embeddings:")
embedding_generator = EmbeddingGenerator(output_dir="../Data/Embeddings")

try:
    embeddings = embedding_generator.load_embeddings(model_name=EMBEDDING_MODEL,
                                                     desc_type=DESCRIPTION_TYPE,
                                                     use_prompt=USE_PROMPT)
    
    activity_descriptions = get_descriptions(DESCRIPTION_TYPE)
    activity_labels = sorted(activity_descriptions.keys())
        
except Exception as e:
    print(f"Error loading embeddings: {e}")
    raise

class_to_embedding_idx = {label: idx for idx, label in enumerate(activity_labels)}

all_embeddings_indexed = np.zeros((max(all_classes) + 1, embeddings.shape[1]))

for cls, idx in class_to_embedding_idx.items():
    all_embeddings_indexed[cls] = embeddings[idx]

unseen_indices = [class_to_embedding_idx[cls] for cls in unseen_adls]
unseen_embeddings = embeddings[unseen_indices]

Loading semantic embeddings:
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)


## Subject–ADL distribution

In [ ]:
file_data = (data[['fileID', 'subject', 'adl']].drop_duplicates()) # Get unique file IDs, subjects, and ADLs
subject_adl = (file_data.groupby(['subject', 'adl'])['fileID']
               .nunique() # Count unique file IDs for each subject and ADL
               .unstack(fill_value=0)) # Unstack to create a matrix of subjects vs ADLs, filling missing values with 0
all_subjects = subject_adl.index.to_numpy()
subject_adl

adl,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
subject,,,,,,,,,,,,,,,,,,,,,
0,5,5,5,5,5,5,5,5,5,5,...,4,5,5,5,5,5,5,5,5,5
1125,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1279,5,4,5,4,5,5,3,3,3,1,...,5,5,5,5,5,5,5,5,5,5
1313,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1324,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1358,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1390,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1396,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5
1405,5,3,5,5,5,5,3,3,3,3,...,5,5,5,5,5,5,5,5,5,5


## Segmentation and feature extraction

In [ ]:
print("Extracting features...")
feature_extractor = FeatureExtractor()

sensor_cols_to_load = [col for col in data.columns if col not in ['timestamp', 'adl', 'session', 'subject', 'fileID']]

X, y, fileIDs, subjects = feature_extractor.extract_features(data=data, method=METHOD, sensor_columns=sensor_cols_to_load,
                                                             window_seconds=WINDOW_SECONDS, overlap_ratio=OVERLAP_RATIO, fs=FS, strategy=STRATEGY)
y = np.asarray(y)
fileIDs = np.asarray(fileIDs)
subjects = np.asarray(subjects)

assert len(X) == len(y) == len(fileIDs) == len(subjects)

Extracting features...

Extracting features: temporal_frequency
  Window: 4.0s
  Overlap: 0.0 (400 stride)
  Sampling rate: 100 Hz

Creating windows per fileID...
  Found 1445 files that are shorter than the window size (30.94% of the available files)
  Created 20103 windows

Extracting features using method: temporal_frequency...
  Processing window 0/20103
                    1000/20103
                    2000/20103
                    3000/20103
                    4000/20103
                    5000/20103
                    6000/20103
                    7000/20103
                    8000/20103
                    9000/20103
                    10000/20103
                    11000/20103
                    12000/20103
                    13000/20103
                    14000/20103
                    15000/20103
                    16000/20103
                    17000/20103
                    18000/20103
                    19000/20103
                    20000/20103
  Featur

# Subject-independent ZSL evaluation

To evaluate generalisation to previously unseen participants, the fixed K=6 ZSL protocol is extended with subject-disjoint training and test sets.

For each subject partition, 30 of the 43 participants are used for training and the remaining 13 are held out for testing. Training contains only the 18 seen ADLs performed by the training subjects, while testing contains only the 6 unseen ADLs performed by the held-out subjects. The training and test sets are disjoint with respect to both subjects and activity classes.

The predefined K=6 activity split, feature representation, semantic descriptions, embedding model, model parameters, and inference procedure are kept unchanged. Only the participant partition varies.

In [ ]:
def run_subject_independent_zsl(X_data, y_data, subjects_data, train_subjects, test_subjects,
                                embeddings_indexed=all_embeddings_indexed, unseen_embeddings_eval=unseen_embeddings,
                                n_runs=N_INFERENCE_RUNS):
    
    train_mask_local = (np.isin(subjects_data, train_subjects) & np.isin(y_data, seen_adls))
    test_mask_local = (np.isin(subjects_data, test_subjects) & np.isin(y_data, unseen_adls))

    y_train_local = y_data[train_mask_local]
    y_test_local = y_data[test_mask_local]

    if set(np.unique(y_train_local)) != set(seen_adls):
        raise ValueError("Training split is missing one or more seen ADLs.")

    if set(np.unique(y_test_local)) != set(unseen_adls):
        raise ValueError("Test split is missing one or more unseen ADLs.")

    model = IP_SAE_MODEL(lambda_reg=LAMBDA, scale_features=True)

    model.fit(X_data[train_mask_local], y_train_local, embeddings_indexed, verbose=0)

    y_pred = model.predict_zsl(X_data[test_mask_local], unseen_embeddings_eval, unseen_adls, n_runs=n_runs)

    ba = balanced_accuracy_score(y_test_local, y_pred)

    return {'balanced_accuracy': ba,
            'model': model,
            'y_true': y_test_local,
            'y_pred': y_pred,
            'train_mask': train_mask_local,
            'test_mask': test_mask_local}

## Subject partitions

The evaluation is repeated over 50 random 30/13 subject partitions while keeping the fixed K=6 activity split unchanged.

The partitions are generated without reference to model performance. A partition is retained only if all 18 seen ADLs are represented in the training set and all 6 unseen ADLs are represented in the test set. Duplicate held-out subject sets are excluded.

In [ ]:
subject_splits = []
used_test_sets = set()

candidate_seed = RANDOM_SEED

while len(subject_splits) < N_SUBJECT_SPLITS:
    rng_split = np.random.default_rng(candidate_seed)

    test_subjects_split = np.sort(rng_split.choice(all_subjects,
                                                   size=num_unseen_subjects,
                                                   replace=False))

    train_subjects_split = np.sort(np.setdiff1d(all_subjects, test_subjects_split))

    train_mask_split = (np.isin(subjects, train_subjects_split) & np.isin(y, seen_adls))
    test_mask_split = (np.isin(subjects, test_subjects_split) & np.isin(y, unseen_adls))

    valid_train = (set(np.unique(y[train_mask_split])) == set(seen_adls))
    valid_test = (set(np.unique(y[test_mask_split])) == set(unseen_adls))

    test_key = tuple(test_subjects_split)

    if (valid_train and valid_test and test_key not in used_test_sets):
        used_test_sets.add(test_key)

        subject_splits.append({'seed': candidate_seed,
                               'train_subjects': train_subjects_split,
                               'test_subjects': test_subjects_split
        })

    candidate_seed += RANDOM_SEED


print(f"Prepared {len(subject_splits)} subject partitions.")
print("Seeds:", [split['seed'] for split in subject_splits])

Prepared 50 subject partitions.
Seeds: [22, 44, 66, 88, 110, 132, 154, 176, 198, 220, 242, 264, 286, 308, 330, 352, 374, 396, 418, 440, 462, 484, 506, 528, 550, 572, 594, 616, 638, 660, 682, 704, 726, 748, 770, 792, 814, 836, 858, 880, 902, 924, 946, 968, 990, 1012, 1034, 1056, 1078, 1100]


## Performance across subject partitions

In [ ]:
subject_results = []
per_class_results = []

for split in subject_splits:

    result = run_subject_independent_zsl(X, y, subjects, split['train_subjects'], split['test_subjects'])

    n_train = result['train_mask'].sum()
    n_test = result['test_mask'].sum()
    n_retained = n_train + n_test
    n_discarded = len(y) - n_retained

    subject_results.append({'seed': split['seed'],
                            'balanced_accuracy': result['balanced_accuracy'],
                            'train_segments': n_train,
                            'test_segments': n_test,
                            'retained_segments': n_retained,
                            'discarded_segments': n_discarded,
                            'retained_%': n_retained / len(y) * 100,
                            'discarded_%': n_discarded / len(y) * 100})

    class_recalls = recall_score(result['y_true'],
                                 result['y_pred'],
                                 labels=unseen_adls,
                                 average=None,
                                 zero_division=0)

    for adl, recall in zip(unseen_adls, class_recalls):
        per_class_results.append({'seed': split['seed'],
                                  'adl': adl,
                                  'recall': recall})

subject_results = pd.DataFrame(subject_results)
subject_results['BA_%'] = (subject_results['balanced_accuracy'] * 100)

per_class_results = pd.DataFrame(per_class_results)
per_class_results['recall_%'] = (per_class_results['recall'] * 100)

subject_results

,seed,balanced_accuracy,train_segments,test_segments,retained_segments,discarded_segments,retained_%,discarded_%,BA_%
0,22,0.700951,10733,1396,12129,7974,60.334278,39.665722,70.095057
1,44,0.700969,9687,1851,11538,8565,57.394419,42.605581,70.096874
2,66,0.551994,9740,1861,11601,8502,57.707805,42.292195,55.199374
3,88,0.578788,8580,2179,10759,9344,53.519375,46.480625,57.878829
4,110,0.615282,10353,1545,11898,8205,59.185196,40.814804,61.528176
5,132,0.496100,9002,2151,11153,8950,55.479282,44.520718,49.609972
6,154,0.603793,9486,2017,11503,8600,57.220315,42.779685,60.379300
7,176,0.587141,9877,1789,11666,8437,58.031140,41.968860,58.714141
8,198,0.628495,9896,1758,11654,8449,57.971447,42.028553,62.849479
9,220,0.539120,9068,2096,11164,8939,55.534000,44.466000,53.911972


### Overall performance

In [ ]:
subject_ba = subject_results['BA_%']

subject_ba_mean = subject_ba.mean()
subject_ba_std = subject_ba.std(ddof=1)
subject_ba_median = subject_ba.median()
subject_ba_q25 = subject_ba.quantile(0.25)
subject_ba_q75 = subject_ba.quantile(0.75)
subject_ba_min = subject_ba.min()
subject_ba_max = subject_ba.max()

print("Subject-independent ZSL, fixed K=6")
print(f"Subject splits: {len(subject_results)}")
print(f"Mean BA: {subject_ba_mean:.2f}%")
print(f"SD: {subject_ba_std:.2f} pp")
print(f"Median BA: {subject_ba_median:.2f}%")
print(f"IQR: {subject_ba_q25:.2f}% - {subject_ba_q75:.2f}%")
print(f"Range: {subject_ba_min:.2f}% - {subject_ba_max:.2f}%")

Subject-independent ZSL, fixed K=6
Subject splits: 50
Mean BA: 58.61%
SD: 7.26 pp
Median BA: 58.46%
IQR: 54.02% - 62.93%
Range: 44.93% - 75.10%


### Per-class performance

In [ ]:
per_class_summary = (per_class_results.groupby('adl')['recall_%']
                     .agg(mean='mean',
                          std='std',
                          min='min',
                          max='max').reset_index())

per_class_summary['Activity'] = per_class_summary['adl'].map(lambda x: adl_dict_raw[str(x)])

per_class_summary['adl'] = pd.Categorical(per_class_summary['adl'], categories=unseen_adls, ordered=True)

per_class_summary = (per_class_summary.sort_values('adl')[['adl', 'Activity', 'mean', 'std', 'min', 'max']].reset_index(drop=True))

per_class_summary.round(2)

,adl,Activity,mean,std,min,max
0,17,type on keyboard,90.80,10.88,38.64,100.00
1,14,stand up,28.95,33.20,0.00,93.48
2,11,put on glasses,61.78,15.09,5.26,90.20
3,9,put on shoe,66.29,23.95,0.00,95.87
4,24,washing dishes,53.54,20.60,7.60,93.43
5,5,brush teeth,50.31,18.95,6.44,92.74


## Additional analyses

The following analyses reuse exactly the same 50 subject partitions to examine whether selected findings from the main study hold under subject-independent evaluation.

### Descriptions

In [ ]:
DESCRIPTION_VARIANTS = ['original',
                        'short',
                        'noisy',
                        'specific',
                        'random_unrelated']


DESCRIPTION_LABELS = {'original': 'Original',
                      'short': 'Short',
                      'noisy': 'Noisy',
                      'specific': 'Specific',
                      'random_unrelated': 'Random'}

description_results = []

for desc_type in DESCRIPTION_VARIANTS:
    print(f"\nDescription: {desc_type}")

    embeddings_desc = (embedding_generator.load_embeddings(model_name=EMBEDDING_MODEL,
                                                           desc_type=desc_type, use_prompt=USE_PROMPT))

    descriptions_desc = get_descriptions(desc_type)

    labels_desc = sorted(descriptions_desc.keys())

    class_to_idx_desc = {label: idx for idx, label in enumerate(labels_desc)}

    indexed_desc = np.zeros((max(all_classes) + 1, embeddings_desc.shape[1]))

    for cls, idx in class_to_idx_desc.items():
        indexed_desc[cls] = (
            embeddings_desc[idx]
        )

    unseen_embeddings_desc = embeddings_desc[[class_to_idx_desc[cls] for cls in unseen_adls]]


    for split in subject_splits:
        result = run_subject_independent_zsl(X, y, subjects, split['train_subjects'], split['test_subjects'],
                                             embeddings_indexed=indexed_desc, unseen_embeddings_eval=unseen_embeddings_desc)

        description_results.append({'description': DESCRIPTION_LABELS[desc_type],
                                    'seed': split['seed'],
                                    'BA_%': result['balanced_accuracy'] * 100})

description_results = pd.DataFrame(description_results)


Description: original
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Description: short
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_short_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Description: noisy
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_noisy_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Description: specific
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_specific_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Description: random_unrelated
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_random_unrelated_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)


In [ ]:
description_summary = (description_results.groupby('description')['BA_%']
                       .agg(mean='mean',
                            std='std',
                            min='min',
                            max='max',
                            median='median').reset_index())

description_order = ['Original',
                     'Short',
                     'Noisy',
                     'Specific',
                     'Random']

description_summary['description'] = pd.Categorical(description_summary['description'],
                                                    categories=description_order, ordered=True)

description_summary = (description_summary.sort_values('description'))

description_summary.round(2)

,description,mean,std,min,max,median
1,Original,58.61,7.26,44.93,75.10,58.46
3,Short,47.31,7.31,30.51,67.62,47.14
0,Noisy,49.49,6.00,38.89,61.09,48.67
4,Specific,44.95,5.84,34.99,57.54,44.71
2,Random,14.89,5.24,5.31,26.13,15.03


### Semantic encoders

In [ ]:
EMBEDDING_MODELS = ['all-MiniLM-L6-v2',
                    'paraphrase-MiniLM-L6-v2',
                    'all-mpnet-base-v2',
                    'hkunlp/instructor-large',
                    'clip-ViT-B/32']

EMBEDDING_LABELS = {'all-MiniLM-L6-v2': 'all-MiniLM-L6-v2',
                    'paraphrase-MiniLM-L6-v2': 'paraphrase-MiniLM-L6-v2',
                    'all-mpnet-base-v2': 'all-mpnet-base-v2',
                    'hkunlp/instructor-large': 'instructor-large',
                    'clip-ViT-B/32': 'CLIP ViT-B/32'}

embedding_model_results = []

for model_name in EMBEDDING_MODELS:
    print(f"\nEmbedding model: {model_name}")

    embeddings_model = embedding_generator.load_embeddings(model_name=model_name,
                                                           desc_type=DESCRIPTION_TYPE,
                                                           use_prompt=USE_PROMPT)

    activity_descriptions_model = get_descriptions(DESCRIPTION_TYPE)

    activity_labels_model = sorted(activity_descriptions_model.keys())

    class_to_idx_model = {label: idx for idx, label in enumerate(activity_labels_model)}

    indexed_model = np.zeros((max(all_classes) + 1, embeddings_model.shape[1]))

    for cls, idx in class_to_idx_model.items():
        indexed_model[cls] = embeddings_model[idx]

    unseen_embeddings_model = embeddings_model[[class_to_idx_model[cls] for cls in unseen_adls]]

    for split in subject_splits:
        result = run_subject_independent_zsl(X, y, subjects, split['train_subjects'], split['test_subjects'],
                                             embeddings_indexed=indexed_model, unseen_embeddings_eval=unseen_embeddings_model)

        embedding_model_results.append({'model': EMBEDDING_LABELS[model_name],
                                        'seed': split['seed'],
                                        'BA_%': (result['balanced_accuracy'] * 100)})

embedding_model_results = pd.DataFrame(embedding_model_results)


Embedding model: all-MiniLM-L6-v2
Loaded embeddings from: ../Data/Embeddings/all_minilm_l6_v2_original_prompt.npz
Model: all-MiniLM-L6-v2
Shape: (24, 384)

Embedding model: paraphrase-MiniLM-L6-v2
Loaded embeddings from: ../Data/Embeddings/paraphrase_minilm_l6_v2_original_prompt.npz
Model: paraphrase-MiniLM-L6-v2
Shape: (24, 384)

Embedding model: all-mpnet-base-v2
Loaded embeddings from: ../Data/Embeddings/all_mpnet_base_v2_original_prompt.npz
Model: all-mpnet-base-v2
Shape: (24, 768)

Embedding model: hkunlp/instructor-large
Loaded embeddings from: ../Data/Embeddings/hkunlp_instructor_large_original_prompt.npz
Model: hkunlp/instructor-large
Shape: (24, 768)

Embedding model: clip-ViT-B/32
Loaded embeddings from: ../Data/Embeddings/clip_vit_b_32_original_prompt.npz
Model: clip-ViT-B/32
Shape: (24, 512)


In [ ]:
embedding_model_summary = (embedding_model_results.groupby('model')['BA_%']
                           .agg(mean='mean',
                                std='std',
                                min='min',
                                max='max',
                                median='median').reset_index().sort_values('mean', ascending=False))

embedding_model_summary.round(2)

,model,mean,std,min,max,median
1,all-MiniLM-L6-v2,58.61,7.26,44.93,75.10,58.46
2,all-mpnet-base-v2,56.33,6.36,40.82,69.43,57.09
4,paraphrase-MiniLM-L6-v2,55.60,4.82,47.66,68.47,55.53
3,instructor-large,50.29,7.32,32.37,75.10,50.26
0,CLIP ViT-B/32,36.06,5.46,20.58,45.60,35.82


### Adaptive trimming

In [ ]:
UNTRIMMED_DATA_PATH = ("../Data/UiS4ADL/Processed/UiS4ADL_100hz_inactivity_removed.csv")

data_untrimmed = pd.read_csv(UNTRIMMED_DATA_PATH)

sensor_cols_untrimmed = [col for col in data_untrimmed.columns
                         if col not in ['timestamp', 'adl', 'session', 'subject', 'fileID' ]]


(X_untrimmed, y_untrimmed, fileIDs_untrimmed, subjects_untrimmed) = feature_extractor.extract_features(data=data_untrimmed,
                                                                                                       method=METHOD,
                                                                                                       sensor_columns=sensor_cols_untrimmed,
                                                                                                       window_seconds=WINDOW_SECONDS,
                                                                                                       overlap_ratio=OVERLAP_RATIO,
                                                                                                       fs=FS,
                                                                                                       strategy=STRATEGY)

X_untrimmed = np.asarray(X_untrimmed)
y_untrimmed = np.asarray(y_untrimmed)

subjects_untrimmed = np.asarray(subjects_untrimmed)


Extracting features: temporal_frequency
  Window: 4.0s
  Overlap: 0.0 (400 stride)
  Sampling rate: 100 Hz

Creating windows per fileID...
  Found 902 files that are shorter than the window size (19.31% of the available files)
  Created 22811 windows

Extracting features using method: temporal_frequency...
  Processing window 0/22811
                    1000/22811
                    2000/22811
                    3000/22811
                    4000/22811
                    5000/22811
                    6000/22811
                    7000/22811
                    8000/22811
                    9000/22811
                    10000/22811
                    11000/22811
                    12000/22811
                    13000/22811
                    14000/22811
                    15000/22811
                    16000/22811
                    17000/22811
                    18000/22811
                    19000/22811
                    20000/22811
                    21000/22811


In [ ]:
trimming_results = []


for split in subject_splits:

    result_trimmed = (run_subject_independent_zsl(X, y, subjects, split['train_subjects'], split['test_subjects']))
    result_untrimmed = (run_subject_independent_zsl(X_untrimmed, y_untrimmed, subjects_untrimmed, split['train_subjects'], split['test_subjects']))
    
    trimming_results.append({'seed': split['seed'],
                             'trimmed_BA_%': result_trimmed['balanced_accuracy'] * 100,
                             'untrimmed_BA_%': result_untrimmed['balanced_accuracy'] * 100,
                             'delta_pp': (result_trimmed['balanced_accuracy'] - result_untrimmed['balanced_accuracy']) * 100})

trimming_results = pd.DataFrame(trimming_results)
trimming_results.round(2)

,seed,trimmed_BA_%,untrimmed_BA_%,delta_pp
0,22,70.10,61.78,8.31
1,44,70.10,51.17,18.92
2,66,55.20,55.01,0.19
3,88,57.88,56.01,1.87
4,110,61.53,58.11,3.41
5,132,49.61,69.13,-19.52
6,154,60.38,58.64,1.74
7,176,58.71,55.76,2.95
8,198,62.85,59.05,3.80
9,220,53.91,49.20,4.71


In [ ]:
trimming_summary = pd.DataFrame({'Condition': ['Untrimmed', 'Trimmed',],
                                 'Mean': [trimming_results['untrimmed_BA_%'].mean(), trimming_results['trimmed_BA_%'].mean()],
                                 'SD': [trimming_results['untrimmed_BA_%'].std(ddof=1), trimming_results['trimmed_BA_%'].std(ddof=1)],
                                 'Min': [trimming_results['untrimmed_BA_%'].min(), trimming_results['trimmed_BA_%'].min()],
                                 'Max': [trimming_results['untrimmed_BA_%'].max(), trimming_results['trimmed_BA_%'].max()]})

trimming_summary.round(2)

,Condition,Mean,SD,Min,Max
0,Untrimmed,54.96,6.09,43.64,69.13
1,Trimmed,58.61,7.26,44.93,75.10


In [ ]:
n_trim_improved = (trimming_results['delta_pp'] > 0).sum()

n_trim_worse = (trimming_results['delta_pp'] < 0).sum()

n_trim_equal = (trimming_results['delta_pp'] == 0).sum()

print(f"Trimming improved BA in {n_trim_improved}/{N_SUBJECT_SPLITS} subject partitions.")

print(f"Trimming reduced BA in {n_trim_worse}/{N_SUBJECT_SPLITS} subject partitions.")

if n_trim_equal:
    print(f"No change in {n_trim_equal}/{N_SUBJECT_SPLITS} partitions.")

Trimming improved BA in 39/50 subject partitions.
Trimming reduced BA in 11/50 subject partitions.
